In [1]:
import torch
print(torch.cuda.is_available())


True


In [2]:
# Cell 1: Environment Setup & Streaming Loader
!pip install datasets transformers timm albumentations opencv-python-headless

import cv2
import numpy as np
import torch
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. DEFINE STRIP-BIAS DATASET ENGINE
class StreamingDualStreamDataset(IterableDataset):
    def __init__(self, hf_dataset):
        self.hf_dataset = hf_dataset

        # Dynamic transformations injected during training runtime
        self.augmentations = A.Compose([
            A.ImageCompression(quality_range=(60, 95), p=0.4),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3),
            A.HorizontalFlip(p=0.5)
        ])

        # Image standardization setups
        self.normalize_global = A.Compose([
            A.Resize(384, 384),
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ToTensorV2()
        ])

        self.normalize_patch = A.Compose([
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ToTensorV2()
        ])

    def __iter__(self):
        for item in self.hf_dataset:
            # Step 1: Extract PIL object from parquet & convert to raw numpy pixels
            # This action natively strips hidden container metadata tags (ICC / EXIF)
            pil_img = item["Image"]
            img = np.array(pil_img.convert("RGB"))

            # Apply dynamic augmentations
            img = self.augmentations(image=img)["image"]
            h, w, _ = img.shape

            # Step 2: Crop a raw native-scale patch (The Scientist microscope)
            patch_size = 128
            if h >= patch_size and w >= patch_size:
                y = np.random.randint(0, h - patch_size)
                x = np.random.randint(0, w - patch_size)
                patch = img[y:y+patch_size, x:x+patch_size]
            else:
                patch = cv2.resize(img, (patch_size, patch_size))

            # Step 3: Package parallel streams
            global_tensor = self.normalize_global(image=img)["image"]
            patch_tensor = self.normalize_patch(image=patch)["image"]

            # Defactify mapping: Label_A handles binary (0=Real, 1=AI)
            label = torch.tensor(item["Label_A"], dtype=torch.long)

            yield global_tensor, patch_tensor, label

# 2. INITIALIZE DIRECT PACKET STREAMING
print("Connecting stream straight to Hugging Face Hub...")
raw_stream = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset", split="train", streaming=True)

# Wrap stream into our dual-stream formatting filter
train_dataset = StreamingDualStreamDataset(raw_stream)
train_loader = DataLoader(train_dataset, batch_size=16)

# Quick connection verification check
for g_batch, p_batch, l_batch in train_loader:
    print("\n[SUCCESS] Pipeline connected and streaming perfectly!")
    print(f"Global Stream Shape  (Art Critic): {g_batch.shape}")
    print(f"Patch Stream Shape   (Scientist) : {p_batch.shape}")
    print(f"Labels Batch Mapping (Real/Fake) : {l_batch}")
    break

Connecting stream straight to Hugging Face Hub...


README.md: 0.00B [00:00, ?B/s]


[SUCCESS] Pipeline connected and streaming perfectly!
Global Stream Shape  (Art Critic): torch.Size([16, 3, 384, 384])
Patch Stream Shape   (Scientist) : torch.Size([16, 3, 128, 128])
Labels Batch Mapping (Real/Fake) : tensor([0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1])


In [3]:
# Cell 2: The Dual-Stream Neural Network Architecture
import torch
import torch.nn as nn
from transformers import AutoModel

class DualStreamDetector(nn.Module):
    def __init__(self):
        super().__init__()
        print("Initializing Dual-Stream Architecture...")

        # ========================================================
        # STREAM A: DEEP SEMANTIC ENCODER (The Art Critic)
        # ========================================================
        # This backbone processes the 384x384 global structural view
        # to spot geometry errors, lighting breaks, and rendering logic.
        self.semantic_backbone = AutoModel.from_pretrained("google/siglip-so400m-patch14-384")

        # Freeze early layers to stabilize features and protect free GPU VRAM
        for param in list(self.semantic_backbone.parameters())[:-12]:
            param.requires_grad = False

        # Extract features from the core vision hidden state sequence (1152 dimensions)
        self.semantic_head = nn.Sequential(
            nn.Linear(1152, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # ========================================================
        # STREAM B: MICRO-PATCH NOISE NETWORK (The Scientist)
        # ========================================================
        # A fast, lightweight Deep CNN processing raw 128x128 patches
        # to learn frequency anomalies and upsampling patterns.
        self.frequency_stream = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), # 128 -> 64

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2), # 64 -> 32

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), # Collapse spatial fields dynamically
            nn.Flatten() # Yields an efficient 128-dimensional footprint vector
        )

        # ========================================================
        # PHASE 3: GATED CROSS-ATTENTION FUSION
        # ========================================================
        # Merges the parallel streams. Learns context weights so compressed
        # noise doesn't corrupt clear geometric choices.
        self.total_features_dim = 512 + 128
        self.gate_layer = nn.Sequential(
            nn.Linear(self.total_features_dim, self.total_features_dim),
            nn.Sigmoid()
        )

        # Final classification output layer (2 classes: Real=0, AI=1)
        self.classifier = nn.Linear(self.total_features_dim, 2)

    def forward(self, global_img, patch_img):
        # 1. Process Stream A (Semantic Global Frame)
        # SigLIP returns a hidden sequence dictionary; extract the pooler output position [0]
        siglip_out = self.semantic_backbone.vision_model(global_img).last_hidden_state[:, 0, :]
        sem_vector = self.semantic_head(siglip_out)

        # 2. Process Stream B (Micro Frequency Patch)
        freq_vector = self.frequency_stream(patch_img)

        # 3. Apply Cross-Stream Gated Fusion
        combined_features = torch.cat((sem_vector, freq_vector), dim=1)
        gate_weights = self.gate_layer(combined_features)
        gated_features = combined_features * gate_weights

        # 4. Generate final raw class outputs
        logits = self.classifier(gated_features)
        return logits

# Instantiate and verify connection mapping to Colab's Free GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DualStreamDetector().to(device)

print(f"\nModel initialized successfully on target hardware device: {device}")

Initializing Dual-Stream Architecture...


config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]


Model initialized successfully on target hardware device: cuda


In [4]:
# Cell 3: Optimized Training and Evaluation Loop
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # We set a step limit per epoch because the streaming dataset is huge (96k images)
    # This prevents your free Colab instance from timing out during execution
    max_steps_per_epoch = 500

    progress_bar = tqdm(enumerate(dataloader), total=max_steps_per_epoch, desc="Training Batches")

    for step, (global_imgs, patch_imgs, labels) in progress_bar:
        if step >= max_steps_per_epoch:
            break

        # Push image matrices straight to the T4 GPU memory allocation
        global_imgs = global_imgs.to(device)
        patch_imgs = patch_imgs.to(device)
        labels = labels.to(device)

        # Reset tracking gradients
        optimizer.zero_grad()

        # Execute the parallel forward pass through your network
        logits = model(global_imgs, patch_imgs)
        loss = criterion(logits, labels)

        # Calculate gradients and optimize the model weights
        loss.backward()
        optimizer.step()

        # Track accuracy metrics
        running_loss += loss.item()
        _, predictions = torch.max(logits, dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_samples += labels.size(0)

        # Dynamic telemetry updates on the progress tracker
        current_accuracy = (correct_predictions / total_samples) * 100
        progress_bar.set_postfix({
            "Loss": f"{running_loss / (step + 1):.4f}",
            "Train Acc": f"{current_accuracy:.2f}%"
        })

# Initialize standard classification loss and optimized learning rate
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

print("Starting training pass on free cloud hardware...")
# Run a verification pass to make sure the data loops and forward steps are working
train_one_epoch(model, train_loader, optimizer, criterion, device)

Starting training pass on free cloud hardware...


Training Batches: 100%|██████████| 500/500 [33:17<00:00,  3.99s/it, Loss=0.1907, Train Acc=92.09%]


In [5]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
# Cell 4: Production-Grade Inference Engine with True Probability Calibration
import torch
import torch.nn.functional as F
from sklearn.isotonic import IsotonicRegression
import numpy as np
import cv2
from PIL import Image

class ProductionDetectorPipeline:
    def __init__(self, trained_model, device):
        self.model = trained_model
        self.device = device
        self.model.eval()

        # Initialize the Isotonic Probability Calibrator
        self.calibrator = IsotonicRegression(out_of_bounds="clip")
        self.is_calibrated = False

        # Define identical standardization rules used in the training pipeline
        self.mean = np.array([0.5, 0.5, 0.5])
        self.std = np.array([0.5, 0.5, 0.5])

    def _preprocess(self, pil_image):
        """Converts raw images to parallel streams while entirely stripping metadata bugs."""
        # Step 1: Force drop EXIF/ICC profile artifacts by going through pure pixel space
        img = np.array(pil_image.convert("RGB"))

        # Step 2: Global Stream (Art Critic Feed)
        global_img = cv2.resize(img, (384, 384))
        global_img = (global_img / 255.0 - self.mean) / self.std
        global_tensor = torch.tensor(global_img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)

        # Step 3: Native Scale Crop (Scientist Feed)
        h, w, _ = img.shape
        patch_size = 128
        if h >= patch_size and w >= patch_size:
            # Use center crop during inference to ensure structural consistency
            y = (h - patch_size) // 2
            x = (w - patch_size) // 2
            patch = img[y:y+patch_size, x:x+patch_size]
        else:
            patch = cv2.resize(img, (patch_size, patch_size))

        patch = (patch / 255.0 - self.mean) / self.std
        patch_tensor = torch.tensor(patch, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)

        return global_tensor.to(self.device), patch_tensor.to(self.device)

    def fit_calibration(self, raw_logits_list, true_labels_list):
        """Standardizes raw neural outputs using an explicit calibration layer."""
        print("Calibrating probability outputs on verification data...")
        raw_probs = np.array(raw_logits_list)
        true_labels = np.array(true_labels_list)
        self.calibrator.fit(raw_probs, true_labels)
        self.is_calibrated = True
        print("Calibration engine locked successfully.")

    def detect(self, pil_image):
        """Calculates a verified, real-world confidence rating for a live web upload."""
        global_tensor, patch_tensor = self._preprocess(pil_image)

        with torch.no_grad():
            logits = self.model(global_tensor, patch_tensor)
            probs = F.softmax(logits, dim=1)
            # Isolate the raw probability score for the AI class (Index 1)
            raw_ai_prob = probs[0, 1].cpu().item()

        # Step 4: Scale the score through our calibration manifold
        if self.is_calibrated:
            calibrated_prob = float(self.calibrator.predict(np.array([raw_ai_prob]))[0])
        else:
            calibrated_prob = raw_ai_prob

        # Step 5: Assign reliable confidence tiers based on mathematical calibration
        if calibrated_prob <= 0.15:
            decision, tier = "Authentic Real Photograph", "HIGH"
        elif calibrated_prob <= 0.45:
            decision, tier = "Likely Real Photograph", "MED"
        elif calibrated_prob < 0.55:
            decision, tier = "Inconclusive (Highly Degraded File)", "LOW"
        elif calibrated_prob <= 0.85:
            decision, tier = "Likely Synthetic AI Generation", "MED"
        else:
            decision, tier = "Verified Synthetic AI Generation", "HIGH"

        return {
            "Decision": decision,
            "Calibrated Probability (AI)": f"{calibrated_prob * 100:.2f}%",
            "Confidence Tier": tier
        }

# Initialize production pipeline container
production_engine = ProductionDetectorPipeline(model, device)
print("Production Inference Engine Compiled.")

Production Inference Engine Compiled.


In [14]:
# Cell 5: Live Test Sandbox (Local Upload or Web URL)
import requests
from io import BytesIO
from PIL import Image
from google.colab import files

def test_image(image_input):
    """
    Passes a PIL Image, local file path, or web URL into the production engine.
    """
    try:
        if isinstance(image_input, Image.Image):
            pil_img = image_input
        elif isinstance(image_input, str) and image_input.startswith("http"):
            response = requests.get(image_input)
            pil_img = Image.open(BytesIO(response.content))
        else:
            pil_img = Image.open(image_input)

        # Run dual-stream inference
        result = production_engine.detect(pil_img)

        print("\n" + "=" * 50)
        print("          DETECTION RESULTS")
        print("=" * 50)
        print(f" Verdict:         {result['Decision']}")
        print(f" AI Probability:  {result['Calibrated Probability (AI)']}")
        print(f" Confidence Tier: ({result['Confidence Tier']})")
        print("=" * 50 + "\n")

    except Exception as e:
        print(f"Error loading image: {e}")


In [13]:
# Cell 5: Live Test Execution
import requests
from io import BytesIO
from PIL import Image

def test_live_image(image_url):
    print("Fetching image from URL...")
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content))
    
    # Run through the Production Inference Engine (Cell 4)
    result = production_engine.detect(img)
    
    print("\n" + "=" * 50)
    print("             MODEL VERDICT")
    print("=" * 50)
    print(f" Target Image:      Hugging Face Official Logo")
    print(f" Expected Ground Truth: Real Photo / Graphic Design (0)")
    print(f" Model Verdict:     {result['Decision']}")
    print(f" AI Probability:    {result['Calibrated Probability (AI)']}")
    print(f" Confidence Tier:   ({result['Confidence Tier']})")
    print("=" * 50)

# Run test on the Hugging Face logo link
target_url = "https://huggingface.co/datasets/huggingface/brand-assets/resolve/main/hf-logo-with-title.png"
test_live_image(target_url)

Fetching image from URL...

             MODEL VERDICT
 Target Image:      Hugging Face Official Logo
 Expected Ground Truth: Real Photo / Graphic Design (0)
 Model Verdict:     Inconclusive (Highly Degraded File)
 AI Probability:    47.12%
 Confidence Tier:   (LOW)
